In [59]:
#basic libraries
import pandas as pd
import numpy as np
import matplotlib as plt

#preprocessing libraries
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

#ML models
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

#ANN libraries
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime


In [60]:
#import data
df = pd.read_csv('credit_cart_data_preprocessed.csv')
df.head()

,Unnamed: 0,Credit Amount,gender,Education level,Marital Status,Age,Y,totall bill statement,total amount paid,avg repayment
0,1,20000,2,2,1,24,1,7704,689,-0.0
1,2,120000,2,2,2,26,1,17077,5000,0.0
2,3,90000,2,2,2,34,0,101653,11018,0.0
3,4,50000,2,2,1,37,0,231334,8388,0.0
4,5,50000,1,2,1,57,0,109339,59049,-0.0


In [61]:
#drop first column
df.drop(['Unnamed: 0'], axis =1, inplace= True)

In [62]:
#split in X and Y
X = df.drop(['Y'], axis= 1)
y = df['Y']

In [63]:
X.describe()

,Credit Amount,gender,Education level,Marital Status,Age,totall bill statement,total amount paid,avg repayment
count,29965.000000,29965.000000,29965.000000,29965.000000,29965.000000,2.996500e+04,2.996500e+04,29965.000000
mean,167442.005006,1.603738,1.853629,1.551877,35.487969,2.701760e+05,3.168778e+04,-0.197364
std,129760.135222,0.489128,0.790411,0.521997,9.219459,3.796744e+05,6.085384e+04,1.012286
min,10000.000000,1.000000,0.000000,0.000000,21.000000,-3.362590e+05,0.000000e+00,-2.000000
25%,50000.000000,1.000000,1.000000,1.000000,28.000000,2.904600e+04,6.700000e+03,-1.000000
50%,140000.000000,2.000000,2.000000,2.000000,34.000000,1.266650e+05,1.440000e+04,0.000000
75%,240000.000000,2.000000,2.000000,2.000000,41.000000,3.429970e+05,3.360000e+04,0.000000
max,1000000.000000,2.000000,6.000000,3.000000,79.000000,5.263883e+06,3.764066e+06,6.000000


In [64]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42)

In [65]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 23972 entries, 27330 to 23654
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Credit Amount          23972 non-null  int64  
 1   gender                 23972 non-null  int64  
 2   Education level        23972 non-null  int64  
 3   Marital Status         23972 non-null  int64  
 4   Age                    23972 non-null  int64  
 5   totall bill statement  23972 non-null  int64  
 6   total amount paid      23972 non-null  int64  
 7   avg repayment          23972 non-null  float64
dtypes: float64(1), int64(7)
memory usage: 1.6 MB


In [66]:
#columns for standardization
train_df_num_features = X_train[['Credit Amount', 'Age', 'totall bill statement','total amount paid', 'avg repayment']]
test_df_num_features = X_test[['Credit Amount', 'Age', 'totall bill statement','total amount paid', 'avg repayment']]


In [67]:
#scalling

preprocessor = StandardScaler()

X_train_scaled = preprocessor.fit_transform(train_df_num_features)
X_test_scaled = preprocessor.transform(test_df_num_features)


In [68]:
#adding scalled features in removing non scaled features
X_train.drop(['Credit Amount', 'Age', 'totall bill statement','total amount paid', 'avg repayment'], axis= 1, inplace= True)
X_test.drop(['Credit Amount', 'Age', 'totall bill statement','total amount paid', 'avg repayment'], axis= 1, inplace= True)

In [69]:
#adding scalled features
X_train = np.concatenate([X_train, X_train_scaled],axis = 1)
X_test = np.concatenate([X_test, X_test_scaled], axis = 1)

X_train.shape

(23972, 8)

In [70]:
#save scalller as pickle
import joblib
joblib.dump(preprocessor, 'scaler.pkl')

['scaler.pkl']

In [71]:
#train ML models

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

def model_evaluator (true, predicted):
    ac = accuracy_score(true, predicted)
    cf = confusion_matrix(true, predicted)
    report = classification_report(true, predicted)

    return ac, cf, report

In [72]:
models = {'Logistic Regression': LogisticRegression(),
          'Random Forest': RandomForestClassifier(),
          'XGB' : XGBClassifier ()}

In [73]:
#iterate and predict
for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train)
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    #evaluate models

    X_train_accuracy, X_train_CF, X_train_Report = model_evaluator(y_train, y_pred_train)
    X_test_accuracy, X_test_CF, X_test_Report = model_evaluator(y_test, y_pred_test)

    print(list(models.keys())[i])

    print('-'*20)
    print('for train data')
    print('-'*20)

    print(f'accuracy_score of the model is: ', X_train_accuracy)
    print(f'confusion matrix of  the model is\n:', X_train_CF)
    print(f'classificaiton report of the model is:\n', X_train_Report)

    print('-'*20)
    print('for test data')
    print('-'*20)
    
    print(f'accuracy_score of the model is: ', X_test_accuracy)
    print(f'confusion matrix of  the model is\n:', X_test_CF)
    print(f'classificaiton report of the model is:\n', X_test_Report)


    print('='*30)

Logistic Regression
--------------------
for train data
--------------------
accuracy_score of the model is:  0.7965125980310362
confusion matrix of  the model is
: [[18308   354]
 [ 4524   786]]
classificaiton report of the model is:
               precision    recall  f1-score   support

           0       0.80      0.98      0.88     18662
           1       0.69      0.15      0.24      5310

    accuracy                           0.80     23972
   macro avg       0.75      0.56      0.56     23972
weighted avg       0.78      0.80      0.74     23972

--------------------
for test data
--------------------
accuracy_score of the model is:  0.7932588019355915
confusion matrix of  the model is
: [[4582   91]
 [1148  172]]
classificaiton report of the model is:
               precision    recall  f1-score   support

           0       0.80      0.98      0.88      4673
           1       0.65      0.13      0.22      1320

    accuracy                           0.79      5993
   macro

ANN modeling

In [74]:
ANN_model = Sequential([
    Dense(64, activation= 'relu', input_shape = (X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation= 'sigmoid')
])

ANN_model.summary()

c:\Users\Shubham\Desktop\Data Science Prep\DS Projects\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 64)             │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,689 (10.50 KB)

 Trainable params: 2,689 (10.50 KB)

 Non-trainable params: 0 (0.00 B)

In [75]:
#compiling models with optimier and loss function
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import binary_crossentropy

optimizer = Adam(learning_rate= 0.01)
loss_fun = binary_crossentropy

ANN_model.compile(optimizer= optimizer, loss= binary_crossentropy, metrics= ['accuracy'])


In [76]:
#setup early stopping
early_stopping_calllback = EarlyStopping(monitor= 'val_loss', patience= 10, restore_best_weights= 'True')

In [77]:
#training model
history = ANN_model.fit(X_train, y_train,
    validation_data = (X_test, y_test),
    epochs = 100,
    callbacks = [early_stopping_calllback]
)

Epoch 1/100
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8003 - loss: 0.4750 - val_accuracy: 0.8039 - val_loss: 0.4704
Epoch 2/100
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8022 - loss: 0.4660 - val_accuracy: 0.8026 - val_loss: 0.4708
Epoch 3/100
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8019 - loss: 0.4635 - val_accuracy: 0.7936 - val_loss: 0.4647
Epoch 4/100
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8022 - loss: 0.4633 - val_accuracy: 0.7993 - val_loss: 0.4656
Epoch 5/100
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8037 - loss: 0.4607 - val_accuracy: 0.8031 - val_loss: 0.4610
Epoch 6/100
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8016 - loss: 0.4614 - val_accuracy: 0.7944 - val_loss: 0.4681
Epoch 7/100
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8040 - loss: 0.4605 - val_accuracy: 0.7929 - val_loss: 0.4619
Epoch 8/100
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8033 - loss: 0.4595 - val_accu

In [78]:
#saving model
ANN_model.save('model.h5')

In [79]:
X_train.shape


(23972, 8)